# Cleanup (Optional)

## Cleanup Memory

In [ ]:
from lab_helpers.lab2_memory import delete_memory

delete_memory(memory_hooks)

### Cleanup: Runtime

In [ ]:
# Delete the AgentCore Runtime
import boto3
from bedrock_agentcore_starter_toolkit import Runtime
from lab_helpers.lab1_guardrails import create_or_get_guardrail_resource
from lab_helpers.lab2_memory import REGION

agentcore_runtime = Runtime()
launch_result = agentcore_runtime.launch()
bedrock_client = boto3.client("bedrock", region_name=REGION)

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id
)
print("Agent runtime deleted:", response["status"])

# Delete the ECR repository
ecr_client = boto3.client("ecr", region_name=REGION)
response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split("/")[1], force=True
)
print("ECR repository deleted:", response["repository"]["repositoryName"])

# Delete the Bedrock Guardrail
guardrail_id, guardrail_version = create_or_get_guardrail_resource()
bedrock_client.delete_guardrail(guardrailIdentifier=guardrail_id)

# Clean up local files
import os

for file in [
    "Dockerfile",
    ".dockerignore",
    ".bedrock_agentcore.yaml",
    "customer_support_agent.py",
    "agent_runtime.py",
]:
    if os.path.exists(file):
        os.unlink(file)
        print(f"Deleted {file}")

print("\n🧹 Cleanup completed!")

### Cleanup: Observability

In [ ]:
# Delete log group and log stream
log_group_name = "agents/customer-support-assistant-logs"  # Your log group name
log_stream_name = "default"  # Your log stream name

import boto3
from botocore.exceptions import ClientError
from lab_helpers.lab2_memory import REGION

logs_client = boto3.client("logs", region_name=REGION)


# Delete log stream first (must be done before deleting log group)
try:
    logs_client.delete_log_stream(
        logGroupName=log_group_name, logStreamName=log_stream_name
    )
    print(f"✅ Log stream '{log_stream_name}' deleted successfully")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFoundException":
        print(f"ℹ️  Log stream '{log_stream_name}' doesn't exist")
    else:
        print(f"❌ Error deleting log stream: {e}")

# Delete log group
try:
    logs_client.delete_log_group(logGroupName=log_group_name)
    print(f"✅ Log group '{log_group_name}' deleted successfully")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFoundException":
        print(f"ℹ️  Log group '{log_group_name}' doesn't exist")
    else:
        print(f"❌ Error deleting log group: {e}")